# 📊 Análisis Exploratorio — Transferencias a Terceros
**Contexto:** Fraude transaccional · Scotiabank Perú · Prevención de Fraude  
**Criterio de extracción Monitor:** Código de transacción = 40 · Tipo de cuenta = 1010  
**Monto mínimo:** S/ 1,200.00  
**Fuente:** Monitor (8750) — Archivos Excel mensuales

---
> **Indicadores de fraude:**  
> `F` = Fraude confirmado · `G` = Buena · `P` = Pendiente · `D` = Descarte  
>
> ⚠️ `COD_TRX=40` y `TIPO_CUENTA=1010` fueron filtros en Monitor — no se analizan (sin variabilidad).

## 0. Librerías

In [ ]:
import polars as pl
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from pathlib import Path
from IPython.display import display

plt.rcParams.update({
    'figure.facecolor': '#0f1117', 'axes.facecolor': '#1a1d27',
    'axes.edgecolor': '#3a3f5c',   'axes.labelcolor': '#e0e0e0',
    'xtick.color': '#e0e0e0',      'ytick.color': '#e0e0e0',
    'text.color': '#e0e0e0',       'grid.color': '#2a2d3e',
    'grid.linestyle': '--',        'grid.alpha': 0.5,
    'figure.dpi': 120,
})
COLOR_FRAUD  = '#ef5350'
COLOR_OK     = '#4fc3f7'
COLOR_ACCENT = '#ffa726'

print(f'Polars  {pl.__version__}')
print(f'Pandas  {pd.__version__}')
print('✅ Librerías cargadas')

## 1. Ingesta — Lectura y unión de los Excel de Monitor

In [ ]:
BASE_DIR = Path(r'C:\Users\s4930359\Data_Herramientas\data\transferencias_terceros')

ARCHIVOS = {
    'Enero' : BASE_DIR / 'monitor_transferencias_enero.xlsx',
    'Marzo' : BASE_DIR / 'monitor_transferencias_marzo.xlsx',
    'Abril' : BASE_DIR / 'monitor_transferencias_abril.xlsx',
    # 'Mes4' : BASE_DIR / 'monitor_transferencias_mes4.xlsx',
}

# Pandas SOLO como lector de .xlsx — Polars no tiene lector nativo de Excel.
# Todo el procesamiento posterior es 100% Polars.
def leer_monitor_excel(path: Path, mes: str) -> pl.DataFrame:
    df_pd = pd.read_excel(path, skiprows=4, dtype=str, header=0, engine='openpyxl')
    df_pd.columns = (
        df_pd.columns
        .str.strip()
        .str.upper()
        .str.replace(r'\s+', '_', regex=True)
        .str.replace(r'[^A-Z0-9_]', '', regex=True)
    )
    return pl.from_pandas(df_pd).with_columns(pl.lit(mes).alias('MES_ORIGEN'))

frames = [leer_monitor_excel(path, mes) for mes, path in ARCHIVOS.items()]
raw = pl.concat(frames, how='diagonal_relaxed')

print(f'Shape total: {raw.shape[0]:,} filas × {raw.shape[1]} columnas')
raw.head(3)

## 2. Inspección de columnas

In [ ]:
for i, col in enumerate(raw.columns, 1):
    print(f'{i:>3}. {col}')

In [ ]:
nulos = (
    raw.select([pl.col(c).is_null().sum().alias(c) for c in raw.columns])
       .unpivot(variable_name='columna', value_name='nulos')
       .with_columns((pl.col('nulos') / raw.height * 100).round(2).alias('pct_nulo'))
       .sort('pct_nulo', descending=True)
)
display(nulos.filter(pl.col('nulos') > 0))

## 3. Mapeo de columnas

> ⚠️ **Completa la clave izquierda** con el nombre exacto que apareció arriba en la sección 2.  
> El alias interno (derecha) no lo toques — el resto del notebook depende de él.

In [ ]:
COL_MAP = {
    # ── COMPLETA EL NOMBRE EXACTO DE MONITOR (clave izquierda) ───────────────
    #
    # Fechas y montos
    'NOMBRE_EXACTO_EN_MONITOR'              : 'FECHA_TRX',          # AAAAMMDD pegado
    'NOMBRE_EXACTO_EN_MONITOR'              : 'HORA_TRX',           # HHMMSS
    'NOMBRE_EXACTO_EN_MONITOR'              : 'MONTO',
    'NOMBRE_EXACTO_EN_MONITOR'              : 'INDICADOR',          # F/G/P/D
    'NOMBRE_EXACTO_EN_MONITOR'              : 'ALERTA',
    #
    # Cuenta origen (cliente que transfiere)
    'NOMBRE_EXACTO_EN_MONITOR'              : 'CUENTA',
    'NOMBRE_EXACTO_EN_MONITOR'              : 'PRIMER_NOMBRE_CLIENTE',
    'NOMBRE_EXACTO_EN_MONITOR'              : 'APELLIDO_CLIENTE',
    #
    # Beneficiario (cuenta destino)
    'NOMBRE_EXACTO_EN_MONITOR'              : 'NOMBRE_BENEFICIARIO', # ACFIP_NOMBRE_CIUDAD
    'NOMBRE_EXACTO_EN_MONITOR'              : 'CUENTA_DESTINO',
    #
    # Dispositivo y canal
    'NOMBRE_EXACTO_EN_MONITOR'              : 'HASH_DISPOSITIVO',   # DNI del dispositivo
    'NOMBRE_EXACTO_EN_MONITOR'              : 'CANAL',              # ATM / MOT
    #
    # Tarjeta: se reconstruye en sección 4 desde estas 2 columnas
    'ACFTARJETA_RESGISTRO_750'              : 'TARJETA_REG',
    'ACFTARJETA_POS_76_DIGITOS'             : 'TARJETA_MID',
    #
    # Saldo y comportamiento
    'ACFSALDO_DISPONIBLE_EN_MONEDA_TRX'    : 'SALDO_DISPONIBLE_RAW',
    'CC_K05_COUNTTMP_TAMANO_COMERCIO'      : 'CNT_MES_PREVIO',     # trx mes anterior
}

cols_a_renombrar = {k: v for k, v in COL_MAP.items() if k in raw.columns}
df = raw.rename(cols_a_renombrar)

print(f'Columnas renombradas: {len(cols_a_renombrar)}')
# Columnas del COL_MAP que NO se encontraron en el dataset (revisar nombre)
no_encontradas = [k for k in COL_MAP if k not in raw.columns]
if no_encontradas:
    print(f'⚠️  No encontradas (revisar nombre): {no_encontradas}')

## 4. Casteo de tipos

In [ ]:
# ── 4.1  MONTO → Float ───────────────────────────────────────────────────────
df = df.with_columns(
    pl.col('MONTO').str.replace_all(',', '').cast(pl.Float64, strict=False)
)

# ── 4.2  SALDO DISPONIBLE → Float ────────────────────────────────────────────
df = (
    df.with_columns(
        pl.col('SALDO_DISPONIBLE_RAW')
          .str.replace_all(',', '')
          .cast(pl.Float64, strict=False)
          .alias('SALDO_DISPONIBLE')
    )
    .drop('SALDO_DISPONIBLE_RAW')
)

# ── 4.3  CNT_MES_PREVIO → Int ────────────────────────────────────────────────
df = df.with_columns(
    pl.col('CNT_MES_PREVIO')
      .str.replace_all(',', '')
      .cast(pl.Int32, strict=False)
)

# ── 4.4  FECHA_TRX → Date (AAAAMMDD pegado) ───────────────────────────────────
df = df.with_columns(
    pl.col('FECHA_TRX').str.strip_chars().str.to_date('%Y%m%d', strict=False)
)

# ── 4.5  HORA_TRX → Time (HHMMSS) ────────────────────────────────────────────
df = df.with_columns(
    pl.col('HORA_TRX').str.strip_chars().str.zfill(6).str.to_time('%H%M%S', strict=False)
)

# ── 4.6  FECHA_HORA combinada → Datetime ─────────────────────────────────────
df = df.with_columns(
    (pl.col('FECHA_TRX').cast(pl.Utf8) + ' ' + pl.col('HORA_TRX').cast(pl.Utf8))
      .str.strptime(pl.Datetime, '%Y-%m-%d %H:%M:%S', strict=False)
      .alias('FECHA_HORA')
)

# ── 4.7  TARJETA reconstruida ─────────────────────────────────────────────────
# TARJETA_REG: primeros 6 + últimos 4  |  TARJETA_MID: 6 dígitos del medio
df = df.with_columns([
    (
        pl.col('TARJETA_REG').str.strip_chars().str.slice(0, 6) +
        pl.col('TARJETA_MID').str.strip_chars() +
        pl.col('TARJETA_REG').str.strip_chars().str.slice(-4)
    ).alias('TARJETA_FULL'),
    pl.col('TARJETA_REG').str.strip_chars().str.slice(0, 6).alias('BIN'),
    pl.col('TARJETA_REG').str.strip_chars().str.slice(-4).alias('TARJETA_LAST4'),
])

# ── 4.8  Limpiar strings clave ────────────────────────────────────────────────
for col in ['INDICADOR', 'CANAL', 'NOMBRE_BENEFICIARIO', 'HASH_DISPOSITIVO']:
    if col in df.columns:
        df = df.with_columns(
            pl.col(col).str.strip_chars().str.to_uppercase().alias(col)
        )

print('✅ Casteo completado')
df.select(['FECHA_TRX','HORA_TRX','MONTO','SALDO_DISPONIBLE',
           'CNT_MES_PREVIO','TARJETA_FULL','BIN','CANAL']).head(5)

## 5. Ingeniería de Variables — Flags & Features

In [ ]:
# ── 5.1  Variable objetivo ────────────────────────────────────────────────────
df = df.with_columns([
    (pl.col('INDICADOR') == 'F').cast(pl.Int8).alias('FLAG_FRAUDE'),
    pl.col('INDICADOR').is_in(['F','G','D']).cast(pl.Int8).alias('FLAG_CLASIFICADO'),
    (pl.col('INDICADOR') == 'P').cast(pl.Int8).alias('FLAG_PENDIENTE'),
])

# ── 5.2  Variables temporales ─────────────────────────────────────────────────
df = df.with_columns([
    pl.col('FECHA_TRX').dt.month().alias('MES'),
    pl.col('FECHA_TRX').dt.day().alias('DIA'),
    pl.col('FECHA_TRX').dt.weekday().alias('DIA_SEMANA'),
    pl.col('FECHA_HORA').dt.hour().alias('HORA'),
    pl.when(pl.col('FECHA_HORA').dt.hour().is_between(0, 5)).then(pl.lit('Madrugada'))
      .when(pl.col('FECHA_HORA').dt.hour().is_between(6, 11)).then(pl.lit('Mañana'))
      .when(pl.col('FECHA_HORA').dt.hour().is_between(12, 17)).then(pl.lit('Tarde'))
      .otherwise(pl.lit('Noche')).alias('FRANJA_HORARIA'),
    (pl.col('FECHA_TRX').dt.weekday() >= 6).cast(pl.Int8).alias('FLAG_FIN_SEMANA'),
    (pl.col('FECHA_TRX').dt.day() >= 26).cast(pl.Int8).alias('FLAG_FIN_MES'),
])

# ── 5.3  Flags de monto ───────────────────────────────────────────────────────
UMBRAL_ALTO    = 5_000.0
UMBRAL_CRITICO = 10_000.0

df = df.with_columns([
    (pl.col('MONTO') >= UMBRAL_ALTO).cast(pl.Int8).alias('FLAG_MONTO_ALTO'),
    (pl.col('MONTO') >= UMBRAL_CRITICO).cast(pl.Int8).alias('FLAG_MONTO_CRITICO'),
])

# ── 5.4  Saldo disponible ─────────────────────────────────────────────────────
df = (
    df.with_columns(
        (pl.col('MONTO') / (pl.col('SALDO_DISPONIBLE') + 1e-9))
          .round(4).alias('RATIO_MONTO_SALDO')
    )
    .with_columns([
        (pl.col('RATIO_MONTO_SALDO') >= 0.80).cast(pl.Int8).alias('FLAG_VACIADO_CUENTA'),
        (pl.col('SALDO_DISPONIBLE') < 200.0).cast(pl.Int8).alias('FLAG_SALDO_BAJO'),
    ])
)

# ── 5.5  Cuenta nueva / sin historia (CC_K05) ────────────────────────────────
# CNT_MES_PREVIO = trx del mes anterior. Bajo → cuenta nueva o mula.
df = df.with_columns([
    (pl.col('CNT_MES_PREVIO').is_null() | (pl.col('CNT_MES_PREVIO') == 0))
      .cast(pl.Int8).alias('FLAG_SIN_HISTORIA'),
    (pl.col('CNT_MES_PREVIO') <= 3)
      .cast(pl.Int8).alias('FLAG_CUENTA_NUEVA'),   # ≤3 trx mes previo
])

# ── 5.6  Beneficiario nulo ────────────────────────────────────────────────────
df = df.with_columns(
    (pl.col('NOMBRE_BENEFICIARIO').is_null() |
     (pl.col('NOMBRE_BENEFICIARIO').str.len_chars() == 0))
      .cast(pl.Int8).alias('FLAG_BENE_NULO')
)

# ── 5.7  Concentración de beneficiario ───────────────────────────────────────
# Cuántas cuentas ORIGEN distintas le transfieren al mismo beneficiario
conc_bene = (
    df.filter(pl.col('FLAG_BENE_NULO') == 0)
      .group_by('NOMBRE_BENEFICIARIO')
      .agg(pl.col('CUENTA').n_unique().alias('N_CUENTAS_ORIGEN_AL_BENE'))
)
df = df.join(conc_bene, on='NOMBRE_BENEFICIARIO', how='left')
# Flag: beneficiario recibe desde muchas cuentas distintas → red de fraude
df = df.with_columns(
    (pl.col('N_CUENTAS_ORIGEN_AL_BENE') >= 5)
      .cast(pl.Int8).alias('FLAG_BENE_CONCENTRADO')
)

# ── 5.8  Hash dispositivo — multiusuario (ATO) ───────────────────────────────
# Cuántas cuentas distintas usaron el mismo dispositivo
hash_multi = (
    df.filter(pl.col('HASH_DISPOSITIVO').is_not_null())
      .group_by('HASH_DISPOSITIVO')
      .agg(pl.col('CUENTA').n_unique().alias('N_CUENTAS_POR_DISPOSITIVO'))
)
df = df.join(hash_multi, on='HASH_DISPOSITIVO', how='left')
# Flag: mismo dispositivo → múltiples cuentas → sospecha de ATO
df = df.with_columns(
    (pl.col('N_CUENTAS_POR_DISPOSITIVO') >= 3)
      .cast(pl.Int8).alias('FLAG_DISPOSITIVO_MULTIUSUARIO')
)

# ── 5.9  Velocidad por cuenta (ventana diaria) ───────────────────────────────
vel_cuenta = (
    df.group_by(['CUENTA','FECHA_TRX'])
      .agg([
          pl.len().alias('VEL_TRX_DIA_CUENTA'),
          pl.col('MONTO').sum().alias('VEL_MONTO_DIA_CUENTA'),
      ])
)
df = df.join(vel_cuenta, on=['CUENTA','FECHA_TRX'], how='left')
df = df.with_columns(
    (pl.col('VEL_TRX_DIA_CUENTA') > 3).cast(pl.Int8).alias('FLAG_VEL_ALTA')
)

# ── 5.10  Velocidad por BIN (ventana diaria) ──────────────────────────────────
vel_bin = (
    df.group_by(['BIN','FECHA_TRX'])
      .agg([
          pl.len().alias('VEL_TRX_DIA_BIN'),
          pl.col('MONTO').sum().alias('VEL_MONTO_DIA_BIN'),
      ])
)
df = df.join(vel_bin, on=['BIN','FECHA_TRX'], how='left')

# ── 5.11  Z-score de monto por cuenta ────────────────────────────────────────
stats_cuenta = (
    df.group_by('CUENTA')
      .agg([pl.col('MONTO').mean().alias('_mu'), pl.col('MONTO').std().alias('_sd')])
)
df = (
    df.join(stats_cuenta, on='CUENTA', how='left')
      .with_columns(
          ((pl.col('MONTO') - pl.col('_mu')) / (pl.col('_sd') + 1e-9))
            .round(3).alias('ZSCORE_MONTO_CUENTA')
      )
      .drop(['_mu','_sd'])
)
df = df.with_columns(
    (pl.col('ZSCORE_MONTO_CUENTA').abs() > 2.0).cast(pl.Int8).alias('FLAG_ANOMALIA_MONTO')
)

print(f'Shape final con features: {df.shape}')
df.select([
    'FECHA_TRX','MONTO','SALDO_DISPONIBLE','RATIO_MONTO_SALDO',
    'CNT_MES_PREVIO','FLAG_CUENTA_NUEVA','FLAG_VACIADO_CUENTA',
    'FLAG_BENE_NULO','FLAG_BENE_CONCENTRADO','FLAG_DISPOSITIVO_MULTIUSUARIO',
    'CANAL','FLAG_FRAUDE'
]).head(5)

## 6. KPIs Ejecutivos

In [ ]:
# Solo clasificadas (F, G, D) — excluye N y P para métricas limpias
df_cal = df.filter(pl.col('FLAG_CLASIFICADO') == 1)

total_trx    = df.height
total_cal    = df_cal.height
total_fraud  = df_cal.filter(pl.col('FLAG_FRAUDE') == 1).height
monto_fraud  = df_cal.filter(pl.col('FLAG_FRAUDE') == 1)['MONTO'].sum()
fraud_rate   = total_fraud / total_cal * 100 if total_cal > 0 else 0
severidad    = monto_fraud / total_fraud if total_fraud > 0 else 0
vaciados_f   = df_cal.filter((pl.col('FLAG_VACIADO_CUENTA')==1) & (pl.col('FLAG_FRAUDE')==1)).height
ato_f        = df_cal.filter((pl.col('FLAG_DISPOSITIVO_MULTIUSUARIO')==1) & (pl.col('FLAG_FRAUDE')==1)).height
nuevas_f     = df_cal.filter((pl.col('FLAG_CUENTA_NUEVA')==1) & (pl.col('FLAG_FRAUDE')==1)).height

print('━' * 65)
print(f'  Total transacciones (dataset)         : {total_trx:>10,}')
print(f'  Transacciones clasificadas             : {total_cal:>10,}')
print(f'  Fraudes confirmados (F)                : {total_fraud:>10,}')
print(f'  Monto total fraudulento (S/)           : {monto_fraud:>13,.2f}')
print(f'  Fraud Rate (sobre clasificadas)        : {fraud_rate:>10.4f} %')
print(f'  Severidad promedio (S/ por caso)       : {severidad:>13,.2f}')
print('─' * 65)
print(f'  Fraudes con vaciado de cuenta (≥80%)  : {vaciados_f:>10,}')
print(f'  Fraudes con dispositivo multiusuario  : {ato_f:>10,}')
print(f'  Fraudes en cuentas nuevas (≤3 trx)    : {nuevas_f:>10,}')
print('━' * 65)

In [ ]:
dist_ind = (
    df.group_by('INDICADOR')
      .agg([pl.len().alias('N'), pl.col('MONTO').sum().alias('MONTO_TOTAL')])
      .with_columns((pl.col('N') / df.height * 100).round(2).alias('PCT'))
      .sort('N', descending=True)
)
display(dist_ind)

## 7. Evolución Mensual

In [ ]:
evol = (
    df_cal.group_by('MES_ORIGEN')
          .agg([
              pl.len().alias('TOTAL_TRX'),
              pl.col('FLAG_FRAUDE').sum().alias('FRAUDES'),
              pl.col('MONTO').filter(pl.col('FLAG_FRAUDE')==1).sum().alias('MONTO_FRAUDE'),
              pl.col('FLAG_VACIADO_CUENTA').filter(pl.col('FLAG_FRAUDE')==1).sum().alias('FRAUDES_VACIADO'),
              pl.col('FLAG_CUENTA_NUEVA').filter(pl.col('FLAG_FRAUDE')==1).sum().alias('FRAUDES_CUENTA_NUEVA'),
          ])
          .with_columns([
              (pl.col('FRAUDES') / pl.col('TOTAL_TRX') * 100).round(4).alias('FRAUD_RATE_PCT'),
              (pl.col('MONTO_FRAUDE') / pl.col('FRAUDES')).round(2).alias('SEVERIDAD'),
          ])
          .sort('MES_ORIGEN')
)
display(evol)

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
fig.suptitle('Evolución Mensual — Transferencias a Terceros', fontsize=13, fontweight='bold')
meses = evol['MES_ORIGEN'].to_list()
for ax, col, color, titulo in zip(
    axes,
    ['FRAUDES','FRAUD_RATE_PCT','MONTO_FRAUDE'],
    [COLOR_FRAUD, COLOR_ACCENT, COLOR_OK],
    ['# Fraudes','Fraud Rate (%)','Monto Fraude (S/)'],
):
    vals = evol[col].to_list()
    bars = ax.bar(meses, vals, color=color, alpha=0.85, edgecolor='white', linewidth=0.5)
    ax.set_title(titulo, fontsize=10)
    ax.set_xlabel('Mes')
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
    ax.bar_label(bars, fmt='%.2f' if col=='FRAUD_RATE_PCT' else '%.0f',
                 fontsize=8, color='white', padding=3)
    ax.grid(axis='y', alpha=0.4)
plt.tight_layout()
plt.show()

## 8. Canal ATM vs MOT

In [ ]:
por_canal = (
    df_cal.group_by('CANAL')
          .agg([
              pl.len().alias('TOTAL'),
              pl.col('FLAG_FRAUDE').sum().alias('FRAUDES'),
              pl.col('MONTO').filter(pl.col('FLAG_FRAUDE')==1).sum().alias('MONTO_FRAUDE'),
              pl.col('MONTO').mean().round(2).alias('TICKET_PROM'),
              pl.col('FLAG_VACIADO_CUENTA').filter(pl.col('FLAG_FRAUDE')==1).sum().alias('VACIADOS_F'),
          ])
          .with_columns(
              (pl.col('FRAUDES') / pl.col('TOTAL') * 100).round(3).alias('FRAUD_RATE')
          )
          .sort('FRAUD_RATE', descending=True)
)
display(por_canal)

# Canal × Franja horaria — heatmap de fraud rate
canal_hora = (
    df_cal.group_by(['CANAL','FRANJA_HORARIA'])
          .agg([pl.len().alias('TOTAL'), pl.col('FLAG_FRAUDE').sum().alias('FRAUDES')])
          .with_columns(
              (pl.col('FRAUDES') / pl.col('TOTAL') * 100).round(3).alias('FRAUD_RATE')
          )
          .sort(['CANAL','FRANJA_HORARIA'])
)

pivot_pd = canal_hora.to_pandas().pivot(index='CANAL', columns='FRANJA_HORARIA', values='FRAUD_RATE').fillna(0)
fig, ax = plt.subplots(figsize=(8, 3))
import matplotlib.colors as mcolors
cmap = mcolors.LinearSegmentedColormap.from_list('fr', ['#1a1d27','#ef5350'])
im = ax.imshow(pivot_pd.values, cmap=cmap, aspect='auto')
ax.set_xticks(range(len(pivot_pd.columns)))
ax.set_xticklabels(pivot_pd.columns)
ax.set_yticks(range(len(pivot_pd.index)))
ax.set_yticklabels(pivot_pd.index)
for i in range(len(pivot_pd.index)):
    for j in range(len(pivot_pd.columns)):
        ax.text(j, i, f'{pivot_pd.values[i,j]:.2f}%', ha='center', va='center', fontsize=9, color='white')
plt.colorbar(im, ax=ax, label='Fraud Rate (%)')
ax.set_title('Fraud Rate: Canal × Franja Horaria', fontsize=11, fontweight='bold')
plt.tight_layout()
plt.show()

## 9. Franja Horaria y Hora del Día

In [ ]:
franja = (
    df_cal.group_by('FRANJA_HORARIA')
          .agg([
              pl.len().alias('TOTAL'),
              pl.col('FLAG_FRAUDE').sum().alias('FRAUDES'),
              pl.col('MONTO').filter(pl.col('FLAG_FRAUDE')==1).sum().alias('MONTO_FRAUDE'),
          ])
          .with_columns((pl.col('FRAUDES') / pl.col('TOTAL') * 100).round(3).alias('FRAUD_RATE'))
          .sort('FRAUD_RATE', descending=True)
)
display(franja)

por_hora = (
    df_cal.group_by('HORA')
          .agg([pl.len().alias('TOTAL'), pl.col('FLAG_FRAUDE').sum().alias('FRAUDES')])
          .with_columns((pl.col('FRAUDES') / pl.col('TOTAL') * 100).round(3).alias('FRAUD_RATE'))
          .sort('HORA')
)

fig, axes = plt.subplots(1, 2, figsize=(16, 4))
fig.suptitle('Análisis Temporal', fontsize=12, fontweight='bold')

rates  = franja['FRAUD_RATE'].to_list()
colors = [COLOR_FRAUD if r == max(rates) else COLOR_OK for r in rates]
bars = axes[0].barh(franja['FRANJA_HORARIA'].to_list(), rates, color=colors, alpha=0.85, edgecolor='white')
axes[0].bar_label(bars, fmt='%.3f%%', fontsize=9, color='white', padding=4)
axes[0].set_title('Fraud Rate por Franja')
axes[0].set_xlabel('Fraud Rate (%)')
axes[0].grid(axis='x', alpha=0.4)

horas = por_hora['HORA'].to_list()
ax2 = axes[1].twinx()
axes[1].bar(horas, por_hora['TOTAL'].to_list(), color=COLOR_OK, alpha=0.4, label='Total trx')
ax2.plot(horas, por_hora['FRAUD_RATE'].to_list(), color=COLOR_FRAUD,
         linewidth=2.5, marker='o', markersize=4, label='Fraud Rate')
axes[1].set_xlabel('Hora')
axes[1].set_ylabel('Volumen', color=COLOR_OK)
ax2.set_ylabel('Fraud Rate (%)', color=COLOR_FRAUD)
axes[1].set_title('Volumen vs Fraud Rate por Hora')
axes[1].set_xticks(range(0, 24))
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 10. Distribución de Montos

In [ ]:
montos_f = df_cal.filter(pl.col('FLAG_FRAUDE')==1)['MONTO'].drop_nulls().to_numpy()
montos_n = df_cal.filter(pl.col('FLAG_FRAUDE')==0)['MONTO'].drop_nulls().to_numpy()

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
fig.suptitle('Distribución de Montos — Transferencias a Terceros', fontsize=12, fontweight='bold')
axes[0].hist(montos_n, bins=40, color=COLOR_OK,   alpha=0.6, label='No Fraude', density=True)
axes[0].hist(montos_f, bins=40, color=COLOR_FRAUD, alpha=0.7, label='Fraude',    density=True)
axes[0].set_xlabel('Monto (S/)'); axes[0].set_ylabel('Densidad')
axes[0].set_title('Distribución (densidad)'); axes[0].legend(); axes[0].grid(alpha=0.3)
axes[1].boxplot([montos_n, montos_f], labels=['No Fraude','Fraude'], patch_artist=True,
                boxprops=dict(facecolor=COLOR_OK, alpha=0.7),
                medianprops=dict(color='white', linewidth=2))
axes[1].set_ylabel('Monto (S/)'); axes[1].set_title('Boxplot'); axes[1].grid(alpha=0.3)
print(f'Mediana Fraude    : S/ {np.median(montos_f):>10,.2f}')
print(f'Mediana No Fraude : S/ {np.median(montos_n):>10,.2f}')
plt.tight_layout(); plt.show()

## 11. Saldo Disponible — Vaciado de Cuenta

In [ ]:
ratio_f = np.clip(df_cal.filter(pl.col('FLAG_FRAUDE')==1)['RATIO_MONTO_SALDO'].drop_nulls().to_numpy(), 0, 1)
ratio_n = np.clip(df_cal.filter(pl.col('FLAG_FRAUDE')==0)['RATIO_MONTO_SALDO'].drop_nulls().to_numpy(), 0, 1)

tramos = [('0-20%',0.,0.2),('20-50%',0.2,0.5),('50-80%',0.5,0.8),('80-100%',0.8,1.0),('>100%',1.0,999.)]
res_t = []
for nombre, lo, hi in tramos:
    sub = df_cal.filter((pl.col('RATIO_MONTO_SALDO')>=lo) & (pl.col('RATIO_MONTO_SALDO')<hi))
    n = sub.height; f = sub.filter(pl.col('FLAG_FRAUDE')==1).height
    res_t.append({'Tramo':nombre,'Total':n,'Fraudes':f,'FraudRate':round(f/n*100,3) if n>0 else 0})
df_tramos = pl.DataFrame(res_t)
display(df_tramos)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
fig.suptitle('Ratio Monto/Saldo Disponible', fontsize=12, fontweight='bold')
axes[0].hist(ratio_n, bins=30, color=COLOR_OK,   alpha=0.6, label='No Fraude', density=True)
axes[0].hist(ratio_f, bins=30, color=COLOR_FRAUD, alpha=0.7, label='Fraude',    density=True)
axes[0].axvline(0.80, color=COLOR_ACCENT, linestyle='--', linewidth=1.5, label='Umbral 80%')
axes[0].set_xlabel('Ratio Monto / Saldo'); axes[0].legend(); axes[0].grid(alpha=0.3)
max_r = df_tramos['FraudRate'].max()
bars = axes[1].bar(df_tramos['Tramo'].to_list(), df_tramos['FraudRate'].to_list(),
                   color=[COLOR_FRAUD if r==max_r else COLOR_OK for r in res_t],
                   alpha=0.85, edgecolor='white')
axes[1].bar_label(bars, fmt='%.3f%%', fontsize=8, color='white', padding=3)
axes[1].set_ylabel('Fraud Rate (%)'); axes[1].grid(axis='y', alpha=0.4)
plt.tight_layout(); plt.show()

## 12. Cuenta Nueva / Sin Historia (CC_K05)

In [ ]:
# Fraud rate por tramo de CNT_MES_PREVIO
tramos_cnt = [
    ('Sin historia (0)',  0,  0),
    ('Muy baja (1-3)',    1,  3),
    ('Baja (4-10)',       4, 10),
    ('Media (11-30)',    11, 30),
    ('Alta (>30)',       31, 9999),
]
res_cnt = []
for nombre, lo, hi in tramos_cnt:
    sub = df_cal.filter(
        pl.col('CNT_MES_PREVIO').is_not_null() &
        (pl.col('CNT_MES_PREVIO') >= lo) & (pl.col('CNT_MES_PREVIO') <= hi)
    )
    n = sub.height; f = sub.filter(pl.col('FLAG_FRAUDE')==1).height
    res_cnt.append({'Tramo':nombre,'Total':n,'Fraudes':f,'FraudRate':round(f/n*100,3) if n>0 else 0})

# Nulos por separado
sub_null = df_cal.filter(pl.col('CNT_MES_PREVIO').is_null())
n = sub_null.height; f = sub_null.filter(pl.col('FLAG_FRAUDE')==1).height
res_cnt.insert(0, {'Tramo':'Nulo','Total':n,'Fraudes':f,'FraudRate':round(f/n*100,3) if n>0 else 0})

df_cnt = pl.DataFrame(res_cnt)
display(df_cnt)

fig, ax = plt.subplots(figsize=(10, 4))
max_r = df_cnt['FraudRate'].max()
bars = ax.bar(df_cnt['Tramo'].to_list(), df_cnt['FraudRate'].to_list(),
              color=[COLOR_FRAUD if r==max_r else COLOR_OK for r in df_cnt['FraudRate'].to_list()],
              alpha=0.85, edgecolor='white')
ax.bar_label(bars, fmt='%.3f%%', fontsize=8, color='white', padding=3)
ax.set_title('Fraud Rate por Historia de Transacciones (mes previo)', fontsize=11, fontweight='bold')
ax.set_ylabel('Fraud Rate (%)')
ax.set_xlabel('Tramo CC_K05 (trx mes anterior)')
ax.grid(axis='y', alpha=0.4)
plt.tight_layout(); plt.show()

## 13. Análisis de Beneficiario

In [ ]:
# 13.1  ¿Los nulos en beneficiario tienen más fraude?
bene_nulo = (
    df_cal.group_by('FLAG_BENE_NULO')
          .agg([pl.len().alias('TOTAL'), pl.col('FLAG_FRAUDE').sum().alias('FRAUDES')])
          .with_columns([
              (pl.col('FRAUDES') / pl.col('TOTAL') * 100).round(3).alias('FRAUD_RATE'),
              pl.col('FLAG_BENE_NULO').replace_strict({0:'Tiene nombre',1:'Sin nombre'}).alias('ESTADO'),
          ])
)
print('Fraud Rate: beneficiario con nombre vs sin nombre:')
display(bene_nulo)

# 13.2  Top beneficiarios por monto fraudulento
top_bene = (
    df_cal.filter((pl.col('FLAG_BENE_NULO')==0) & (pl.col('FLAG_FRAUDE')==1))
          .group_by('NOMBRE_BENEFICIARIO')
          .agg([
              pl.len().alias('N_FRAUDES'),
              pl.col('MONTO').sum().alias('MONTO_FRAUDE'),
              pl.col('CUENTA').n_unique().alias('N_CUENTAS_ORIGEN'),
          ])
          .sort('MONTO_FRAUDE', descending=True)
)
print('\nTop 15 beneficiarios con mayor monto fraudulento recibido:')
display(top_bene.head(15))

In [ ]:
# 13.3  Beneficiarios concentrados (reciben de muchas cuentas)
bene_concentrados = (
    df_cal.filter(pl.col('FLAG_BENE_NULO')==0)
          .group_by('NOMBRE_BENEFICIARIO')
          .agg([
              pl.col('CUENTA').n_unique().alias('N_CUENTAS_ORIGEN'),
              pl.len().alias('TOTAL_TRX'),
              pl.col('FLAG_FRAUDE').sum().alias('FRAUDES'),
              pl.col('MONTO').sum().alias('MONTO_TOTAL'),
          ])
          .with_columns(
              (pl.col('FRAUDES') / pl.col('TOTAL_TRX') * 100).round(3).alias('FRAUD_RATE')
          )
          .filter(pl.col('N_CUENTAS_ORIGEN') >= 3)
          .sort('N_CUENTAS_ORIGEN', descending=True)
)
print('Beneficiarios que reciben desde 3+ cuentas distintas (posible red):')
display(bene_concentrados.head(20))

## 14. Análisis de Dispositivo — Detección de ATO

In [ ]:
# Dispositivos que operan múltiples cuentas
dispositivos_riesgo = (
    df_cal.filter(pl.col('HASH_DISPOSITIVO').is_not_null())
          .group_by('HASH_DISPOSITIVO')
          .agg([
              pl.col('CUENTA').n_unique().alias('N_CUENTAS'),
              pl.len().alias('TOTAL_TRX'),
              pl.col('FLAG_FRAUDE').sum().alias('FRAUDES'),
              pl.col('MONTO').filter(pl.col('FLAG_FRAUDE')==1).sum().alias('MONTO_FRAUDE'),
              pl.col('FECHA_TRX').min().alias('PRIMERA_TRX'),
              pl.col('FECHA_TRX').max().alias('ULTIMA_TRX'),
          ])
          .with_columns(
              (pl.col('FRAUDES') / pl.col('TOTAL_TRX') * 100).round(3).alias('FRAUD_RATE')
          )
          .filter(pl.col('N_CUENTAS') >= 2)
          .sort('N_CUENTAS', descending=True)
)
print(f'Dispositivos que operan 2+ cuentas distintas: {dispositivos_riesgo.height:,}')
print('Top 20 dispositivos de mayor riesgo (ATO):')
display(dispositivos_riesgo.head(20))

# Distribución de N_CUENTAS por dispositivo
fig, ax = plt.subplots(figsize=(9, 4))
bins_data = dispositivos_riesgo['N_CUENTAS'].to_numpy()
ax.hist(bins_data, bins=min(20, len(set(bins_data))),
        color=COLOR_FRAUD, alpha=0.85, edgecolor='white')
ax.set_xlabel('N° de cuentas distintas por dispositivo')
ax.set_ylabel('N° de dispositivos')
ax.set_title('Distribución de Cuentas por Dispositivo — Riesgo ATO', fontsize=11, fontweight='bold')
ax.grid(axis='y', alpha=0.4)
plt.tight_layout(); plt.show()

## 15. Análisis por BIN

In [ ]:
por_bin = (
    df_cal.group_by('BIN')
          .agg([
              pl.len().alias('TOTAL'),
              pl.col('FLAG_FRAUDE').sum().alias('FRAUDES'),
              pl.col('MONTO').filter(pl.col('FLAG_FRAUDE')==1).sum().alias('MONTO_FRAUDE'),
              pl.col('MONTO').mean().round(2).alias('TICKET_PROM'),
          ])
          .with_columns(
              (pl.col('FRAUDES') / pl.col('TOTAL') * 100).round(3).alias('FRAUD_RATE')
          )
          .filter(pl.col('FRAUDES') > 0)
          .sort('MONTO_FRAUDE', descending=True)
)
print('Top 15 BINs por monto fraudulento:')
display(por_bin.head(15))

top10 = por_bin.head(10)
fig, ax = plt.subplots(figsize=(12, 4))
bars = ax.bar(top10['BIN'].to_list(), top10['MONTO_FRAUDE'].to_list(),
              color=COLOR_FRAUD, alpha=0.85, edgecolor='white')
ax.bar_label(bars, fmt='S/ %.0f', fontsize=7, color='white', padding=3)
ax.set_title('Top 10 BINs — Monto Fraudulento', fontsize=11, fontweight='bold')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
ax.grid(axis='y', alpha=0.4)
plt.tight_layout(); plt.show()

## 16. Propuesta de Reglas — Base para Monitor

In [ ]:
reglas = [
    # ── Reglas simples ────────────────────────────────────────────────────────
    ('R01: Monto Alto (≥5,000)',              pl.col('FLAG_MONTO_ALTO') == 1),
    ('R02: Monto Crítico (≥10,000)',          pl.col('FLAG_MONTO_CRITICO') == 1),
    ('R03: Velocidad Alta (>3 trx/día)',      pl.col('FLAG_VEL_ALTA') == 1),
    ('R04: Anomalía Monto (|z|>2)',           pl.col('FLAG_ANOMALIA_MONTO') == 1),
    ('R05: Madrugada (0-5h)',                 pl.col('FRANJA_HORARIA') == 'Madrugada'),
    ('R06: Vaciado de cuenta (≥80%)',         pl.col('FLAG_VACIADO_CUENTA') == 1),
    ('R07: Cuenta nueva (≤3 trx previas)',    pl.col('FLAG_CUENTA_NUEVA') == 1),
    ('R08: Beneficiario nulo',                pl.col('FLAG_BENE_NULO') == 1),
    ('R09: Dispositivo multiusuario (ATO)',   pl.col('FLAG_DISPOSITIVO_MULTIUSUARIO') == 1),
    ('R10: Beneficiario concentrado (≥5c)',   pl.col('FLAG_BENE_CONCENTRADO') == 1),
    # ── Reglas combinadas ─────────────────────────────────────────────────────
    ('R11: Vaciado + Madrugada',              (pl.col('FLAG_VACIADO_CUENTA')==1) & (pl.col('FRANJA_HORARIA')=='Madrugada')),
    ('R12: Vaciado + Vel Alta',               (pl.col('FLAG_VACIADO_CUENTA')==1) & (pl.col('FLAG_VEL_ALTA')==1)),
    ('R13: Cuenta nueva + Monto Alto',        (pl.col('FLAG_CUENTA_NUEVA')==1) & (pl.col('FLAG_MONTO_ALTO')==1)),
    ('R14: MOT + Madrugada + Vaciado',        (pl.col('CANAL')=='MOT') & (pl.col('FRANJA_HORARIA')=='Madrugada') & (pl.col('FLAG_VACIADO_CUENTA')==1)),
    ('R15: ATO + Monto Alto',                 (pl.col('FLAG_DISPOSITIVO_MULTIUSUARIO')==1) & (pl.col('FLAG_MONTO_ALTO')==1)),
    ('R16: Cuenta nueva + Vaciado + Madrugada', (pl.col('FLAG_CUENTA_NUEVA')==1) & (pl.col('FLAG_VACIADO_CUENTA')==1) & (pl.col('FRANJA_HORARIA')=='Madrugada')),
]

resultados = []
for nombre, condicion in reglas:
    sub = df_cal.filter(condicion)
    n_alertas = sub.height
    n_fraudes = sub.filter(pl.col('FLAG_FRAUDE')==1).height
    precision = n_fraudes / n_alertas * 100 if n_alertas > 0 else 0
    recall    = n_fraudes / total_fraud * 100 if total_fraud > 0 else 0
    monto_cap = sub.filter(pl.col('FLAG_FRAUDE')==1)['MONTO'].sum()
    resultados.append({
        'Regla'      : nombre,
        'Alertas'    : n_alertas,
        'Fraudes'    : n_fraudes,
        'Precision%' : round(precision, 2),
        'Recall%'    : round(recall, 2),
        'Monto_Cap'  : round(monto_cap, 2) if monto_cap else 0,
    })

df_reglas = pl.DataFrame(resultados).sort('Precision%', descending=True)
print('📋 Evaluación de Reglas Propuestas:')
display(df_reglas)

fig, ax = plt.subplots(figsize=(15, 5))
x = np.arange(len(resultados))
w = 0.35
ax.bar(x-w/2, df_reglas['Precision%'].to_list(), w, color=COLOR_FRAUD,  alpha=0.85, label='Precisión (%)', edgecolor='white')
ax.bar(x+w/2, df_reglas['Recall%'].to_list(),    w, color=COLOR_ACCENT, alpha=0.85, label='Recall (%)',    edgecolor='white')
ax.set_xticks(x)
ax.set_xticklabels(df_reglas['Regla'].to_list(), rotation=40, ha='right', fontsize=7)
ax.set_ylabel('%')
ax.set_title('Precisión vs Recall por Regla', fontsize=12, fontweight='bold')
ax.legend(); ax.grid(axis='y', alpha=0.4)
plt.tight_layout(); plt.show()

## 17. Export

In [ ]:
OUTPUT_PATH = Path(r'C:\Users\s4930359\Data_Herramientas\data\transferencias_terceros')
OUTPUT_PATH.mkdir(parents=True, exist_ok=True)

# Dataset completo enriquecido
df.write_parquet(OUTPUT_PATH / 'transferencias_terceros_enriquecido.parquet')

# Solo fraudes para revisión manual
(
    df_cal.filter(pl.col('FLAG_FRAUDE')==1)
          .sort('MONTO', descending=True)
          .to_pandas()
          .to_excel(OUTPUT_PATH / 'fraudes_transferencias_terceros.xlsx', index=False)
)

# Tabla de reglas
df_reglas.to_pandas().to_excel(OUTPUT_PATH / 'evaluacion_reglas.xlsx', index=False)

# Beneficiarios concentrados (red de fraude)
bene_concentrados.to_pandas().to_excel(OUTPUT_PATH / 'beneficiarios_red_fraude.xlsx', index=False)

# Dispositivos de riesgo (ATO)
dispositivos_riesgo.to_pandas().to_excel(OUTPUT_PATH / 'dispositivos_ato.xlsx', index=False)

print('✅ Archivos exportados:')
print('   → transferencias_terceros_enriquecido.parquet')
print('   → fraudes_transferencias_terceros.xlsx')
print('   → evaluacion_reglas.xlsx')
print('   → beneficiarios_red_fraude.xlsx')
print('   → dispositivos_ato.xlsx')

---
## 📝 Resumen del Análisis

| Dimensión | Sección | Variable clave |
|---|---|---|
| **Evolución mensual** | 7 | Fraud Rate, Severidad |
| **Canal** | 8 | ATM vs MOT × Franja horaria |
| **Temporalidad** | 9 | Hora pico de fraude |
| **Montos** | 10 | Mediana fraude vs no fraude |
| **Saldo / vaciado** | 11 | RATIO_MONTO_SALDO |
| **Historia cuenta** | 12 | CNT_MES_PREVIO — mule accounts |
| **Red beneficiarios** | 13 | N_CUENTAS_ORIGEN_AL_BENE |
| **ATO dispositivo** | 14 | N_CUENTAS_POR_DISPOSITIVO |
| **BIN** | 15 | Exposición por BIN |
| **Reglas** | 16 | R14, R15, R16 — mayor precisión esperada |

> **Próximo paso:** Presentar reglas R14–R16 al especialista con la tabla de Precisión/Recall/Monto capturado.